# core

> Workflow-level route handlers for status, reset, and source retrieval

In [ ]:
#| default_exp routes.core

In [ ]:
#| export
from pathlib import Path

from starlette.responses import FileResponse, Response

from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import StructureDecompWorkflow

# Debug flag for audio serving tracing (set False in production)
DEBUG_AUDIO = False

## Core Route Handlers

These handlers manage workflow status and lifecycle.

In [ ]:
#| export
def _handle_current_status(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess  # FastHTML session object
):  # Appropriate UI component based on current state
    """Return current workflow status - determines what to show."""
    session_id = get_session_id(sess)
    
    # Check for in-progress workflow (via server-side state store)
    workflow_state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)
    current_step = workflow.state_store.get_current_step(workflow.config.workflow_id, session_id)
    
    # Restore external paths from state to SourceService (fixes persistence across restarts)
    step_states = workflow_state.get("step_states", {})
    selection_state = step_states.get("selection", {})
    external_db_paths = selection_state.get("external_db_paths", [])
    if external_db_paths:
        workflow.source_service.set_external_paths(external_db_paths)
    
    # If there's existing state, resume the workflow
    if current_step or workflow_state:
        return workflow._stepflow_router.start(request, sess)
    
    # Start fresh
    return workflow._stepflow_router.start(request, sess)

In [ ]:
#| export
def _handle_reset(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess  # FastHTML session object
):  # StepFlow start view
    """Reset workflow and return to start."""
    session_id = get_session_id(sess)
    workflow.state_store.clear_state(workflow.config.workflow_id, session_id)
    return workflow._stepflow_router.start(request, sess)

In [ ]:
#| export
def _handle_get_sources(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    provider_id: str = None,  # Optional plugin name filter
    limit: int = 50  # Maximum number of results
):  # JSON response with transcription sources
    """Get available transcription sources."""
    transcriptions = workflow.source_service.query_transcriptions(
        provider_id=provider_id,
        limit=limit
    )
    return {"transcriptions": transcriptions}

## Combined Step Handlers

Handlers for the combined Phase 2 step: chrome switching and audio serving.

In [ ]:
#| export
async def _handle_switch_chrome(
    workflow:StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
) -> tuple:  # OOB swaps for shared chrome containers
    """Switch shared chrome content based on active column.

    Reads `active_column` from form data and returns OOB swaps for
    toolbar, controls, and footer. Full implementation in sub-step 4d.
    """
    # Stub — returns empty tuple until zone switching is wired in 4d
    return ()

In [ ]:
#| export
def _handle_audio_src(
    workflow:StructureDecompWorkflow,  # The workflow instance
    sess,  # FastHTML session object
):  # Audio file response or 404
    """Serve the audio file for the current alignment session."""
    session_id = get_session_id(sess)

    if DEBUG_AUDIO:
        print(f"[AUDIO_SRC] Serving audio for session: {session_id}")

    state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)
    align_state = state.get("step_states", {}).get("alignment", {})
    media_path = align_state.get("media_path")

    if DEBUG_AUDIO:
        print(f"[AUDIO_SRC] media_path from state: {media_path}")
        if media_path:
            print(f"[AUDIO_SRC] file exists: {Path(media_path).is_file()}")

    if not media_path or not Path(media_path).is_file():
        if DEBUG_AUDIO:
            print(f"[AUDIO_SRC] Returning 404 - media_path missing or file not found")
        return Response(status_code=404)

    if DEBUG_AUDIO:
        print(f"[AUDIO_SRC] Returning FileResponse for: {media_path}")

    return FileResponse(str(media_path), media_type="audio/mpeg")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()